# Iteración, recursos y concurrencia

**Unidad 1 · Sesión 3**

Generadores, context managers y concurrencia con `asyncio`.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Generadores y consumo bajo demanda |
| 2 | `with`, decoradores y `@contextmanager` |
| 3 | `async`/`await` y tareas concurrentes |
| 4 | Tiempos límite y limpieza |

Al terminar, podrás reconocer cuándo conviene procesar valores uno a uno,
mantener un recurso abierto solo durante su uso y solapar esperas independientes.

## 1. Generadores y consumo bajo demanda

Una lista conserva sus elementos. Un **iterador** mantiene la posición de un
recorrido; `next()` solicita el siguiente valor. Cuando se agota, no vuelve al
inicio. Una función con `yield` crea un generador: produce un valor cuando el
consumidor lo pide y continúa desde donde se quedó.

In [ ]:
values = iter([18.0, 27.0, 32.0])
print(next(values))
print(list(values))
print(list(values))  # The iterator is exhausted.

In [ ]:
from collections.abc import Iterator


def readings(values: list[float]) -> Iterator[float]:
    for value in values:
        print(f"Reading {value}")
        yield value


stream = readings([18.0, 27.0, 32.0])
print("Generator created")
print(next(stream))
print(list(stream))

Crear `stream` no ejecutó el cuerpo de `readings`. El primer `next()` produjo
un solo valor; `list(stream)` consumió el resto. Esto permite recorrer datos
sin construir antes una lista con todos los resultados. Si el consumidor usa
`list()`, entonces sí materializa lo que queda.

### Ejercicio 1 · Números pares bajo demanda

Escribe `even_numbers(stop: int)`, una función generadora que produzca los
números pares desde cero hasta antes de `stop`. Comprueba que
`list(even_numbers(7))` sea `[0, 2, 4, 6]` y que `list(even_numbers(0))`
sea una lista vacía. Usa `yield`; no construyas una lista dentro de la función.

In [ ]:
# Define even_numbers and check both cases.

## 2. Recursos con `with`

Un archivo debe cerrarse al terminar de usarlo. `with` delimita ese periodo:
al salir del bloque, Python llama al cierre del recurso, también si dentro
ocurre un error. El archivo se puede recorrer línea por línea.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")

    with path.open(encoding="utf-8") as file:
        first_line = next(file).strip()
        print(first_line)

    assert file.closed

### Salida por error

El cierre no depende de que el bloque termine normalmente. El error puede
atenderse fuera de `with` y el archivo ya estará cerrado.

In [ ]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("invalid\n", encoding="utf-8")

    try:
        with path.open(encoding="utf-8") as file:
            raise ValueError("Invalid reading")
    except ValueError as error:
        print(error)

    assert file.closed

### Antes de `@contextmanager`: qué hace un decorador

Un decorador recibe una función y devuelve otra que añade comportamiento.
La forma `@announce` aplica `announce` a la función definida debajo. Aquí
solo observamos esa idea; no hace falta crear una biblioteca de decoradores.

In [ ]:
def announce(function):
    def wrapper():
        print("Starting")
        return function()

    return wrapper


@announce
def show_message():
    print("Reading complete")


show_message()

### Crear un context manager con `@contextmanager`

`@contextmanager` permite escribir un context manager sin definir una clase.
La función llega hasta `yield` al entrar en `with`; el bloque `finally` se
ejecuta al salir. En esta demostración, el recurso es un archivo en memoria.

In [ ]:
from contextlib import contextmanager
from io import StringIO


@contextmanager
def open_text(text: str) -> Iterator[StringIO]:
    file = StringIO(text)
    try:
        yield file
    finally:
        file.close()


with open_text("north\nsouth\n") as file:
    print(next(file).strip())

assert file.closed

### Ejercicio 2 · Cierre de un archivo

Crea un archivo temporal con dos líneas. Ábrelo con `with`, lee solo la primera
y comprueba que esté cerrado después del bloque. Luego provoca un `ValueError`
dentro de otro bloque `with`, atiéndelo fuera del bloque y comprueba de nuevo
que el archivo quedó cerrado.

In [ ]:
# Use TemporaryDirectory and Path; check file.closed in both situations.

## 3. Concurrencia con `asyncio`

Una corrutina se define con `async def`. `await` permite esperar una operación
sin impedir que otras tareas listas avancen. Aquí `asyncio.sleep()` representa
una espera de entrada/salida. En Jupyter y Colab podemos usar `await`
directamente en una celda.

In [ ]:
import asyncio


async def fetch_reading(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    await asyncio.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


print(await fetch_reading("north", 0.05))

### Esperar en secuencia o crear tareas

Dos `await` consecutivos terminan una lectura antes de iniciar la otra.
`TaskGroup` inicia tareas independientes y espera a que todas terminen al
salir de `async with`. Las tareas pueden completarse en otro orden; cada
variable conserva el resultado de la tarea que se le asignó.

In [ ]:
from time import perf_counter


start = perf_counter()
north = await fetch_reading("north", 0.08)
south = await fetch_reading("south", 0.04)
sequential_time = perf_counter() - start
print(north, south, f"Sequential: {sequential_time:.2f} s")

start = perf_counter()
async with asyncio.TaskGroup() as group:
    north_task = group.create_task(fetch_reading("north", 0.08))
    south_task = group.create_task(fetch_reading("south", 0.04))
concurrent_time = perf_counter() - start
print(north_task.result(), south_task.result(), f"Concurrent: {concurrent_time:.2f} s")

La segunda ejecución suele tardar menos porque las esperas se solapan.
La duración exacta depende del entorno; lo importante es observar el orden
de los mensajes y que ambos resultados están disponibles al terminar el
grupo. Esto ayuda con esperas independientes de red o disco; no acelera por
sí solo un cálculo intensivo.

### Ejercicio 3 · Dos lecturas concurrentes

Crea dos tareas con `TaskGroup`: `fetch_reading("north", 0.06)` y
`fetch_reading("west", 0.02)`. Después del bloque, comprueba con `assert`
los dos resultados. Explica por qué `west` puede anunciarse primero aunque
la tarea de `north` se haya creado antes.

In [ ]:
# Create two tasks, read their results after TaskGroup, and check both tuples.

## 4. Tiempo límite y limpieza

Una operación puede tardar demasiado. `asyncio.timeout()` establece un tiempo
máximo para el bloque. Si se agota, se produce `TimeoutError`. La corrutina
puede usar `finally` para limpiar su estado incluso cuando se cancela la espera.

In [ ]:
events: list[str] = []


async def wait_for_update() -> None:
    events.append("started")
    try:
        await asyncio.Event().wait()
    finally:
        events.append("cleaned up")


try:
    async with asyncio.timeout(0.02):
        await wait_for_update()
except TimeoutError:
    print("Deadline reached")

assert events == ["started", "cleaned up"]
print(events)

### Ejercicio 4 · Comprobar la limpieza

Escribe una corrutina que añada `"started"` a una lista, espere más de 0.05 s
y añada `"cleaned up"` en `finally`. Llámala dentro de
`asyncio.timeout(0.01)`, atiende `TimeoutError` y comprueba que ambos mensajes
quedaron en la lista. Después prueba una espera corta que sí termine a tiempo.

In [ ]:
# Define a coroutine with try/finally; check a timeout and a completed wait.

## Cierre

- Un generador produce valores cuando se solicitan; consumirlo por completo
  puede volver a ocupar memoria para guardar los resultados.
- `with` delimita el uso de un recurso; `@contextmanager` permite definir
  esa entrada y salida mediante una función.
- `asyncio` ayuda a solapar esperas independientes. Usa un tiempo límite
  cuando una espera no deba continuar indefinidamente.

En un archivo `.py` se inicia el event loop desde una función principal con
`asyncio.run(main())`; en esta notebook usamos `await` directamente.

## Referencias

- [Tutorial de Python: generadores](https://docs.python.org/3/tutorial/classes.html#generators).
- [Python: `contextlib`](https://docs.python.org/3/library/contextlib.html).
- [Python: corrutinas y tareas](https://docs.python.org/3/library/asyncio-task.html).
- [Python: ejecutar `asyncio`](https://docs.python.org/3/library/asyncio-runner.html).